<a href="https://colab.research.google.com/github/zchnee/1111/blob/main/scenario1_confusable_subset_pipeline_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Scenario 1: Confusable-Class Subset Construction (with Dataset Download Fallbacks)

Goal: construct confusable-class subsets from **FGVCAircraft**, **StanfordCars**, and **OxfordPets**.

Pipeline:

```text
download / mount datasets
→ run zero-shot CLIP on full dataset
→ collect high-confidence wrong predictions
→ mine class-pair confusions
→ export subset CSVs
```


In [ ]:
# =========================
# 0. Colab setup
# =========================

from google.colab import drive
import os
from pathlib import Path

# correct check: MyDrive exists means mounted
if os.path.exists("/content/drive/MyDrive"):
    print("Drive already mounted.")
else:
    drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/cs543_tpt_scenario1")
RAW_ROOT = PROJECT_ROOT / "raw_datasets"
IMAGEFOLDER_ROOT = PROJECT_ROOT / "imagefolder_datasets"
OUTPUT_ROOT = PROJECT_ROOT / "confusable_subsets"

for p in [PROJECT_ROOT, RAW_ROOT, IMAGEFOLDER_ROOT, OUTPUT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_ROOT:", RAW_ROOT)
print("IMAGEFOLDER_ROOT:", IMAGEFOLDER_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

print("\nRAW_ROOT contents:")
for p in RAW_ROOT.iterdir():
    print(p)


Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/cs543_tpt_scenario1
RAW_ROOT: /content/drive/MyDrive/cs543_tpt_scenario1/raw_datasets
IMAGEFOLDER_ROOT: /content/drive/MyDrive/cs543_tpt_scenario1/imagefolder_datasets
OUTPUT_ROOT: /content/drive/MyDrive/cs543_tpt_scenario1/confusable_subsets

RAW_ROOT contents:
/content/drive/MyDrive/cs543_tpt_scenario1/raw_datasets/fgvc-aircraft-2013b.tar.gz
/content/drive/MyDrive/cs543_tpt_scenario1/raw_datasets/fgvc-aircraft-2013b
/content/drive/MyDrive/cs543_tpt_scenario1/raw_datasets/oxford-iiit-pet
/content/drive/MyDrive/cs543_tpt_scenario1/raw_datasets/stanford_cars


In [ ]:
# =========================
# 1. Install dependencies
# =========================

!pip -q install ftfy regex tqdm open_clip_torch kaggle scipy pandas matplotlib

import os
import shutil
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from PIL import Image


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 78.0 MB/s eta 0:00:00


## 2. Dataset download

`FGVCAircraft` and `OxfordPets` can be downloaded by torchvision.

`torchvision.datasets.StanfordCars(download=True)` fails with:

```text
ValueError: The original URL is broken so the StanfordCars dataset cannot be downloaded anymore.
```

So the notebook uses a fallback with kaggle.


In [ ]:
# =========================
# 2. Download / prepare datasets
# =========================

from torchvision.datasets import FGVCAircraft, OxfordIIITPet

DATASET_SPLIT = "test"  # for subset mining, test is fine; train also works

def safe_download_aircraft(raw_root):
    print("\n[FGVCAircraft]")
    split = "test" if DATASET_SPLIT == "test" else "train"
    ds = FGVCAircraft(root=raw_root, split=split, annotation_level="variant", download=True)
    print("Loaded FGVCAircraft:", len(ds))
    return ds

def safe_download_pets(raw_root):
    print("\n[OxfordPets]")
    split = "test" if DATASET_SPLIT == "test" else "trainval"
    ds = OxfordIIITPet(root=raw_root, split=split, target_types="category", download=True)
    print("Loaded OxfordPets:", len(ds))
    return ds

aircraft_ds = safe_download_aircraft(RAW_ROOT)
pets_ds = safe_download_pets(RAW_ROOT)



[FGVCAircraft]
Loaded FGVCAircraft: 3333

[OxfordPets]


100%|██████████| 792M/792M [00:27<00:00, 28.5MB/s]
100%|██████████| 19.2M/19.2M [00:01<00:00, 15.9MB/s]


Loaded OxfordPets: 3669


In [ ]:
# =========================
# 2b. StanfordCars fallback
# =========================

!pip install -q kaggle

import os
from getpass import getpass
from pathlib import Path

os.environ["KAGGLE_API_TOKEN"] = getpass("Kaggle API token: ")

!kaggle datasets download -d rickyyyyyyy/torchvision-stanford-cars -p "{RAW_ROOT}" --unzip


KeyboardInterrupt: Interrupted by user

## 3. Convert datasets to a simple manifest

Instead of forcing every dataset into ImageFolder immediately, this notebook creates a unified CSV manifest:

```text
dataset,image_path,label_id,class_name
```

That is easier for subset construction and avoids StanfordCars folder-format problems.


In [ ]:
from pathlib import Path

CARS_ROOT = Path("/content/drive/MyDrive/cs543_tpt_scenario1/raw_datasets/stanford_cars")

print("CARS_ROOT exists:", CARS_ROOT.exists())
print("CARS_ROOT:", CARS_ROOT)

print("Files found:")
for p in CARS_ROOT.rglob("*annos*"):
    print(p)

CARS_ROOT exists: False
CARS_ROOT: /content/drive/MyDrive/cs543_tpt_scenario1/raw_datasets/stanford_cars
Files found:


In [ ]:
# =========================
# 3. Build manifests
# =========================

import pandas as pd
from tqdm.auto import tqdm

def get_image_path(ds, i):
    if hasattr(ds, "_image_files"):
        return ds._image_files[i]
    if hasattr(ds, "_images"):
        return ds._images[i]
    if hasattr(ds, "images"):
        return ds.images[i]
    return ds[i][0]

def get_label(ds, i):
    if hasattr(ds, "_labels"):
        return ds._labels[i]
    if hasattr(ds, "targets"):
        return ds.targets[i]
    return ds[i][1]

def get_class_name(ds, label):
    if hasattr(ds, "classes"):
        return ds.classes[label]
    if hasattr(ds, "_classes"):
        return ds._classes[label]
    return str(label)

def manifest_from_torchvision_dataset(ds, dataset_name, out_csv):
    rows = []

    for i in tqdm(range(len(ds)), desc=f"Manifest {dataset_name}"):
        path = get_image_path(ds, i)
        label = int(get_label(ds, i))
        class_name = get_class_name(ds, label)

        rows.append({
            "dataset": dataset_name,
            "image_id": i,
            "path": str(path),
            "label": label,
            "class_name": class_name,
        })

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(f"Saved {dataset_name} manifest:", out_csv, len(df))
    return df

In [ ]:
# =========================
# Build FULL Aircraft + Pets manifests
# =========================

from torchvision.datasets import FGVCAircraft, OxfordIIITPet
import pandas as pd

# Aircraft full = train + val + test
aircraft_splits = ["train", "val", "test"]
aircraft_manifests = []

for split in aircraft_splits:
    ds = FGVCAircraft(
        root=RAW_ROOT,
        split=split,
        annotation_level="variant",
        download=True
    )

    df = manifest_from_torchvision_dataset(
        ds,
        "fgvc_aircraft",
        f"{OUTPUT_ROOT}/manifest_fgvc_aircraft_{split}.csv"
    )
    df["split"] = split
    aircraft_manifests.append(df)

aircraft_manifest = pd.concat(aircraft_manifests, ignore_index=True)
aircraft_manifest.to_csv(f"{OUTPUT_ROOT}/manifest_fgvc_aircraft.csv", index=False)
print("Saved full aircraft manifest:", len(aircraft_manifest))


# Pets full = trainval + test
pet_splits = ["trainval", "test"]
pet_manifests = []

for split in pet_splits:
    ds = OxfordIIITPet(
        root=RAW_ROOT,
        split=split,
        target_types="category",
        download=True
    )

    df = manifest_from_torchvision_dataset(
        ds,
        "oxford_pets",
        f"{OUTPUT_ROOT}/manifest_oxford_pets_{split}.csv"
    )
    df["split"] = split
    pet_manifests.append(df)

pets_manifest = pd.concat(pet_manifests, ignore_index=True)
pets_manifest.to_csv(f"{OUTPUT_ROOT}/manifest_oxford_pets.csv", index=False)
print("Saved full pets manifest:", len(pets_manifest))

Manifest fgvc_aircraft:   0%|          | 0/3334 [00:00<?, ?it/s]

Saved fgvc_aircraft manifest: /content/drive/MyDrive/cs543_tpt_scenario1/confusable_subsets/manifest_fgvc_aircraft_train.csv 3334


Manifest fgvc_aircraft:   0%|          | 0/3333 [00:00<?, ?it/s]

Saved fgvc_aircraft manifest: /content/drive/MyDrive/cs543_tpt_scenario1/confusable_subsets/manifest_fgvc_aircraft_val.csv 3333


Manifest fgvc_aircraft:   0%|          | 0/3333 [00:00<?, ?it/s]

Saved fgvc_aircraft manifest: /content/drive/MyDrive/cs543_tpt_scenario1/confusable_subsets/manifest_fgvc_aircraft_test.csv 3333
Saved full aircraft manifest: 10000


Manifest oxford_pets:   0%|          | 0/3680 [00:00<?, ?it/s]

Saved oxford_pets manifest: /content/drive/MyDrive/cs543_tpt_scenario1/confusable_subsets/manifest_oxford_pets_trainval.csv 3680


Manifest oxford_pets:   0%|          | 0/3669 [00:00<?, ?it/s]

Saved oxford_pets manifest: /content/drive/MyDrive/cs543_tpt_scenario1/confusable_subsets/manifest_oxford_pets_test.csv 3669
Saved full pets manifest: 7349


In [ ]:
# =========================
# 3b. Optional StanfordCars manifest
# =========================
# Build StanfordCars manifest from Kaggle-downloaded dataset

import scipy.io
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

RAW_ROOT = Path("/content/drive/MyDrive/cs543_tpt_scenario1/raw_datasets")
CARS_ROOT = RAW_ROOT / "stanford_cars"
OUTPUT_ROOT = Path("/content/drive/MyDrive/cs543_tpt_scenario1/confusable_subsets")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def find_file(root, name):
    matches = list(Path(root).rglob(name))
    if len(matches) == 0:
        raise FileNotFoundError(f"Cannot find {name} under {root}")
    return matches[0]

def find_dir(root, name):
    matches = [p for p in Path(root).rglob(name) if p.is_dir()]
    if len(matches) == 0:
        raise FileNotFoundError(f"Cannot find folder {name} under {root}")
    return matches[0]

def build_cars_manifest(cars_root, split="train", out_csv=None):
    cars_root = Path(cars_root)

    if split == "train":
        anno_path = find_file(cars_root, "cars_train_annos.mat")
        img_dir = find_dir(cars_root, "cars_train")
    else:
        anno_path = find_file(cars_root, "cars_test_annos_withlabels.mat")
        img_dir = find_dir(cars_root, "cars_test")

    meta_path = find_file(cars_root, "cars_meta.mat")

    print("Annotation:", anno_path)
    print("Images:", img_dir)
    print("Meta:", meta_path)

    annos = scipy.io.loadmat(anno_path)["annotations"][0]
    class_names = scipy.io.loadmat(meta_path)["class_names"][0]

    rows = []
    for i, a in enumerate(tqdm(annos, desc=f"Manifest stanford_cars {split}")):
        img_name = a["fname"][0]
        label = int(a["class"][0][0]) - 1
        class_name = class_names[label][0]

        rows.append({
            "dataset": "stanford_cars",
            "image_id": i,
            "path": str(img_dir / img_name),
            "label": label,
            "class_name": class_name,
        })

    df = pd.DataFrame(rows)

    if out_csv:
        df.to_csv(out_csv, index=False)
        print(f"Saved: {out_csv}, n={len(df)}")

    return df

def build_cars_full_manifest(cars_root, out_csv=None):
    train_df = build_cars_manifest(cars_root, split="train", out_csv=None)
    train_df["split"] = "train"

    test_df = build_cars_manifest(cars_root, split="test", out_csv=None)
    test_df["split"] = "test"

    full_df = pd.concat([train_df, test_df], ignore_index=True)

    if out_csv:
        full_df.to_csv(out_csv, index=False)
        print(f"Saved full StanfordCars manifest: {out_csv}, n={len(full_df)}")

    return full_df

cars_manifest = build_cars_full_manifest(
    CARS_ROOT,
    out_csv=OUTPUT_ROOT / "manifest_stanford_cars.csv"
)

cars_manifest.head()



Annotation: /content/drive/MyDrive/cs543_tpt_scenario1/raw_datasets/stanford_cars/devkit/cars_train_annos.mat
Images: /content/drive/MyDrive/cs543_tpt_scenario1/raw_datasets/stanford_cars/cars_train
Meta: /content/drive/MyDrive/cs543_tpt_scenario1/raw_datasets/stanford_cars/devkit/cars_meta.mat


Manifest stanford_cars train:   0%|          | 0/8144 [00:00<?, ?it/s]

Annotation: /content/drive/MyDrive/cs543_tpt_scenario1/raw_datasets/stanford_cars/cars_test_annos_withlabels.mat
Images: /content/drive/MyDrive/cs543_tpt_scenario1/raw_datasets/stanford_cars/cars_test
Meta: /content/drive/MyDrive/cs543_tpt_scenario1/raw_datasets/stanford_cars/devkit/cars_meta.mat


Manifest stanford_cars test:   0%|          | 0/8041 [00:00<?, ?it/s]

Saved full StanfordCars manifest: /content/drive/MyDrive/cs543_tpt_scenario1/confusable_subsets/manifest_stanford_cars.csv, n=16185


,dataset,image_id,path,label,class_name,split
0,stanford_cars,0,/content/drive/MyDrive/cs543_tpt_scenario1/raw...,13,Audi TTS Coupe 2012,train
1,stanford_cars,1,/content/drive/MyDrive/cs543_tpt_scenario1/raw...,2,Acura TL Sedan 2012,train
2,stanford_cars,2,/content/drive/MyDrive/cs543_tpt_scenario1/raw...,90,Dodge Dakota Club Cab 2007,train
3,stanford_cars,3,/content/drive/MyDrive/cs543_tpt_scenario1/raw...,133,Hyundai Sonata Hybrid Sedan 2012,train
4,stanford_cars,4,/content/drive/MyDrive/cs543_tpt_scenario1/raw...,105,Ford F-450 Super Duty Crew Cab 2012,train


## 4. Zero-shot CLIP over full dataset

This cell runs CLIP once on the full dataset and records:

```text
true class
predicted class
confidence
correct / incorrect
```

This is the only model step needed before mining confusable pairs.


In [ ]:
# =========================
# 4. Load CLIP
# =========================

import open_clip

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "ViT-B-16"
PRETRAINED = "openai"

model, _, preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME, pretrained=PRETRAINED, device=DEVICE
)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.eval()

print("Device:", DEVICE)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Device: cuda


In [ ]:
# =========================
# 4b. Dataset and zero-shot helper
# =========================

class ManifestImageDataset(torch.utils.data.Dataset):
    def __init__(self, manifest_df, preprocess):
        self.df = manifest_df.reset_index(drop=True)
        self.preprocess = preprocess
        self.classes = sorted(self.df["class_name"].unique())
        self.class_to_id = {c: i for i, c in enumerate(self.classes)}
        # remap label ids to 0..K-1 based on class_name to be safe
        self.df["label_id_remap"] = self.df["class_name"].map(self.class_to_id)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        return self.preprocess(img), int(row["label_id_remap"]), row["image_path"]

def build_text_features(class_names, template="a photo of a {}."):
    prompts = [template.format(c.replace("_", " ")) for c in class_names]
    text = tokenizer(prompts).to(DEVICE)
    with torch.no_grad():
        text_features = model.encode_text(text)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    return text_features, prompts

@torch.no_grad()
def run_zeroshot_manifest(manifest_df, dataset_name, batch_size=64, template="a photo of a {}."):
    ds = ManifestImageDataset(manifest_df, preprocess)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2)
    text_features, prompts = build_text_features(ds.classes, template=template)

    rows = []
    for images, labels, paths in tqdm(loader, desc=f"Zero-shot {dataset_name}"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        image_features = model.encode_image(images)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        logits = 100.0 * image_features @ text_features.T
        probs = logits.softmax(dim=-1)
        conf, pred = probs.max(dim=-1)

        for pth, y, yhat, cf in zip(paths, labels.cpu().numpy(), pred.cpu().numpy(), conf.cpu().numpy()):
            rows.append({
                "dataset": dataset_name,
                "image_path": pth,
                "true_label_id": int(y),
                "true_class": ds.classes[int(y)],
                "pred_label_id": int(yhat),
                "pred_class": ds.classes[int(yhat)],
                "confidence": float(cf),
                "correct": bool(int(y) == int(yhat)),
            })

    out = pd.DataFrame(rows)
    out_csv = f"{OUTPUT_ROOT}/zeroshot_{dataset_name}.csv"
    out.to_csv(out_csv, index=False)
    print("Saved:", out_csv)
    print("Accuracy:", out["correct"].mean())
    return out


In [ ]:
# =========================
# 4c. Run zero-shot on available datasets
# =========================

def normalize_manifest(df):
    df = df.copy()
    if "image_path" not in df.columns and "path" in df.columns:
        df = df.rename(columns={"path": "image_path"})
    return df

aircraft_manifest = normalize_manifest(aircraft_manifest)
pets_manifest = normalize_manifest(pets_manifest)

if cars_manifest is not None:
    cars_manifest = normalize_manifest(cars_manifest)


RUN_AIRCRAFT = True
RUN_PETS = True
RUN_CARS = cars_manifest is not None

zs_results = {}

if RUN_AIRCRAFT:
    zs_results["fgvc_aircraft"] = run_zeroshot_manifest(
        aircraft_manifest, "fgvc_aircraft", template="a photo of a {}, a type of aircraft."
    )

if RUN_PETS:
    zs_results["oxford_pets"] = run_zeroshot_manifest(
        pets_manifest, "oxford_pets", template="a photo of a {}, a type of pet."
    )
# first two done last to do
if RUN_CARS:
    zs_results["stanford_cars"] = run_zeroshot_manifest(
        cars_manifest, "stanford_cars", template="a photo of a {}, a type of car."
    )
else:
    print("Skipping StanfordCars zero-shot because no valid cars manifest is available.")


Zero-shot fgvc_aircraft:   0%|          | 0/157 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/cs543_tpt_scenario1/confusable_subsets/zeroshot_fgvc_aircraft.csv
Accuracy: 0.2048


Zero-shot oxford_pets:   0%|          | 0/115 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/cs543_tpt_scenario1/confusable_subsets/zeroshot_oxford_pets.csv
Accuracy: 0.8367124778881481


Zero-shot stanford_cars:   0%|          | 0/253 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/cs543_tpt_scenario1/confusable_subsets/zeroshot_stanford_cars.csv
Accuracy: 0.5704046957059006


## 5. Mine high-confidence wrong class pairs

For each true class `A`, look at wrong predictions into class `B`.

A pair `(A → B)` is confusable when:
- many images from `A` are predicted as `B`;
- the predictions have high average confidence;
- the source class has low accuracy.

This directly implements the proposal idea: identify class pairs with **high-confidence but low-accuracy predictions**, then restrict evaluation to these pairs.


In [ ]:
# =========================
# 5. Confusable pair mining (improved)
# =========================

def mine_confusable_pairs(
    zs_df,
    min_wrong_count=5,
    min_mean_conf=0.4,
    top_k=100,
    use_quantile=False,
    quantile=0.9
):
    wrong = zs_df[zs_df["correct"] == False].copy()
    if len(wrong) == 0:
        return pd.DataFrame()

    # source class accuracy
    acc_by_class = zs_df.groupby("true_class")["correct"].mean().rename("source_class_acc")

    pairs = (
        wrong.groupby(["true_class", "pred_class"])
        .agg(
            wrong_count=("image_path", "count"),
            mean_confidence=("confidence", "mean"),
            max_confidence=("confidence", "max"),
        )
        .reset_index()
    )

    pairs = pairs.merge(acc_by_class, on="true_class", how="left")

    # basic filtering
    pairs = pairs[
        (pairs["wrong_count"] >= min_wrong_count) &
        (pairs["mean_confidence"] >= min_mean_conf)
    ].copy()

    # scoring
    pairs["confusion_score"] = (
        pairs["wrong_count"] *
        pairs["mean_confidence"] *
        (1.0 - pairs["source_class_acc"])
    )

    pairs = pairs.sort_values("confusion_score", ascending=False)

    # choose selection strategy
    if use_quantile:
        threshold = pairs["confusion_score"].quantile(quantile)
        pairs = pairs[pairs["confusion_score"] >= threshold]
        print(f"Using quantile threshold: {quantile}, score >= {threshold:.4f}")
    else:
        pairs = pairs.head(top_k)
        print(f"Using top_k={top_k}")

    return pairs

for name in ["fgvc_aircraft", "oxford_pets", "stanford_cars"]:
    csv_path = OUTPUT_ROOT / f"zeroshot_{name}.csv"
    if csv_path.exists():
        zs_results[name] = pd.read_csv(csv_path)
        print("Loaded:", name, len(zs_results[name]))
    else:
        print("Missing:", csv_path)

# run for all datasets
pair_tables = {}

for name, df in zs_results.items():
    print("\n====================")
    print("Dataset:", name)

    pairs = mine_confusable_pairs(
        df,
        min_wrong_count=5,
        min_mean_conf=0.4,
        top_k=100,           # increased from 30
        use_quantile=False   # set True if you want adaptive selection
    )

    pair_tables[name] = pairs

    out_csv = f"{OUTPUT_ROOT}/confusable_pairs_{name}.csv"
    pairs.to_csv(out_csv, index=False)

    print("Total pairs:", len(pairs))
    display(pairs.head(10))
    print("Saved:", out_csv)

Loaded: fgvc_aircraft 10000
Loaded: oxford_pets 7349
Loaded: stanford_cars 16185

Dataset: fgvc_aircraft
Using top_k=100
Total pairs: 1


,true_class,pred_class,wrong_count,mean_confidence,max_confidence,source_class_acc,confusion_score
894,DC-3,C-47,6,0.524763,0.842456,0.6,1.25943


Saved: /content/drive/MyDrive/cs543_tpt_scenario1/confusable_subsets/confusable_pairs_fgvc_aircraft.csv

Dataset: oxford_pets
Using top_k=100
Total pairs: 42


,true_class,pred_class,wrong_count,mean_confidence,max_confidence,source_class_acc,confusion_score
31,Birman,Ragdoll,174,0.712638,0.930449,0.095,112.219085
104,Persian,Maine Coon,51,0.443354,0.753842,0.260,16.732191
103,Persian,British Shorthair,46,0.426239,0.855326,0.260,14.509171
105,Persian,Ragdoll,46,0.423151,0.784629,0.260,14.404073
26,Bengal,Abyssinian,45,0.619720,0.971591,0.705,8.226785
51,Egyptian Mau,Bengal,41,0.608063,0.890476,0.700,7.479174
20,American Pit Bull Terrier,Staffordshire Bull Terrier,31,0.447406,0.867038,0.570,5.963919
116,Russian Blue,British Shorthair,38,0.478353,0.828541,0.720,5.089677
10,American Pit Bull Terrier,American Bulldog,19,0.553117,0.919641,0.570,4.518963
5,American Bulldog,Boxer,26,0.582900,0.874918,0.775,3.409965


Saved: /content/drive/MyDrive/cs543_tpt_scenario1/confusable_subsets/confusable_pairs_oxford_pets.csv

Dataset: stanford_cars
Using top_k=100
Total pairs: 71


,true_class,pred_class,wrong_count,mean_confidence,max_confidence,source_class_acc,confusion_score
170,Audi TT RS Coupe 2012,Audi TTS Coupe 2012,76,0.513994,0.827156,0.000000,39.063538
704,Dodge Sprinter Cargo Van 2009,Mercedes-Benz Sprinter Van 2012,66,0.602075,0.902600,0.101266,35.712943
166,Audi TT Hatchback 2011,Audi TTS Coupe 2012,73,0.492061,0.787523,0.049383,34.146631
531,Chevrolet Tahoe Hybrid SUV 2012,Chevrolet Avalanche Crew Cab 2012,61,0.521488,0.880933,0.081081,29.231538
122,Audi RS 4 Convertible 2008,Audi S5 Convertible 2012,61,0.503633,0.865744,0.082192,28.196547
261,BMW X5 SUV 2007,BMW X3 SUV 2012,57,0.450090,0.588850,0.096386,23.182355
612,Dodge Caliber Wagon 2007,Dodge Caliber Wagon 2012,60,0.509580,0.765034,0.250000,22.931080
535,Chevrolet TrailBlazer SS 2009,Chevrolet Avalanche Crew Cab 2012,54,0.401470,0.610927,0.100000,19.511443
1278,Spyker C8 Coupe 2009,Spyker C8 Convertible 2009,48,0.538793,0.810291,0.247059,19.472596
1220,Plymouth Neon Coupe 1999,Eagle Talon Hatchback 1998,49,0.410441,0.864663,0.090909,18.283286


Saved: /content/drive/MyDrive/cs543_tpt_scenario1/confusable_subsets/confusable_pairs_stanford_cars.csv


## 6. Export restricted subset CSVs

For each selected pair `(A, B)`, export all images whose ground-truth class is either `A` or `B`.

These CSVs become the input to later TPT / DynaPrompt experiments.


In [ ]:
# =========================
# 6. Save final confusable subsets as image folders
# =========================

import shutil
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

OUTPUT_ROOT = Path(OUTPUT_ROOT)
IMAGE_SUBSET_ROOT = Path("/content/drive/MyDrive/cs543_tpt_scenario1/final_confusable_subset_images")
IMAGE_SUBSET_ROOT.mkdir(parents=True, exist_ok=True)

manifest_files = {
    # "fgvc_aircraft": OUTPUT_ROOT / "manifest_fgvc_aircraft.csv",
    # "oxford_pets": OUTPUT_ROOT / "manifest_oxford_pets.csv",
    "stanford_cars": OUTPUT_ROOT / "manifest_stanford_cars.csv",
}

for name, manifest_path in manifest_files.items():
    pair_path = OUTPUT_ROOT / f"confusable_pairs_{name}.csv"

    if not manifest_path.exists() or not pair_path.exists():
        print("Skipping:", name)
        continue

    manifest = pd.read_csv(manifest_path)
    pairs = pd.read_csv(pair_path)

    if "image_path" not in manifest.columns and "path" in manifest.columns:
        manifest = manifest.rename(columns={"path": "image_path"})

    selected_classes = set(pairs["true_class"]).union(set(pairs["pred_class"]))

    subset = manifest[manifest["class_name"].isin(selected_classes)].copy()

    # save subset csv too
    subset_csv = OUTPUT_ROOT / f"final_subset_{name}.csv"
    subset.to_csv(subset_csv, index=False)

    # copy images into class folders
    out_dir = IMAGE_SUBSET_ROOT / name
    out_dir.mkdir(parents=True, exist_ok=True)

    for _, row in tqdm(subset.iterrows(), total=len(subset), desc=f"Copy {name}"):
        src = Path(row["image_path"])
        class_name = str(row["class_name"]).replace("/", "_")
        dst_dir = out_dir / class_name
        dst_dir.mkdir(parents=True, exist_ok=True)

        dst = dst_dir / src.name
        if not dst.exists():
            shutil.copy2(src, dst)

    print(f"\n{name}")
    print("Selected classes:", len(selected_classes))
    print("Saved subset CSV:", subset_csv)
    print("Saved images to:", out_dir)

Copy stanford_cars:   0%|          | 0/7303 [00:00<?, ?it/s]


stanford_cars
Selected classes: 90
Saved subset CSV: /content/drive/MyDrive/cs543_tpt_scenario1/confusable_subsets/final_subset_stanford_cars.csv
Saved images to: /content/drive/MyDrive/cs543_tpt_scenario1/final_confusable_subset_images/stanford_cars


## 7. Next step

The output files under `scenario1_confusable_subsets/` are now ready for later TPT experiments.


In [ ]:
!ls "/content/drive/MyDrive/cs543_tpt_scenario1/final_confusable_subset_images"

fgvc_aircraft  oxford_pets  stanford_cars
